# Evaluate code

In [1]:
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact
from langsmith import evaluate, aevaluate

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found qwen/qwen3-coder-480b-a35b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(


## Criando dataset no LangSmith

In [2]:
from datasets import load_dataset
from tqdm import tqdm

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)


INFO:numexpr.utils:NumExpr defaulting to 16 threads.
Problems: 100%|██████████| 164/164 [00:00<00:00, 4236.07problem/s]


In [ ]:
from langsmith import Client

client_langsmith = Client()


tamanho_amostra = 5

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(data_set_code_langsmith)), tamanho_amostra)
for index in index_aleatorios:
    data_aleatorios.append(data_set_code_langsmith[index])


# Create dataset if it doesn't exist
if not client_langsmith.has_dataset(dataset_name=dataset_name):
    dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name, 
        description="The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models."
    )
    
    client_langsmith.create_examples(dataset_id=dataset.id, examples=data_aleatorios)

ERROR:opentelemetry.sdk._shared_internal:Exception while exporting Span.
Traceback (most recent call last):
  File "c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jefer\AppData\Roaming\uv\python\cpython-3.12.0-windows-x86_64-none\Lib\http\client.py", line 1411, in getresponse
    response.begin()
  File "C:\Users\jefer\AppData\Roaming\uv\python\cpython-3.12.0-windows-x86_64-none\Lib\http\client.py", line 324, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jefer\AppData\Roaming\uv\pyth

In [4]:
data_aleatorios

[{'inputs': {'aswer_code': '\ndef simplify(x, n):\n    """Your task is to implement a function that will simplify the expression\n    x * n. The function returns True if x * n evaluates to a whole number and False\n    otherwise. Both x and n, are string representation of a fraction, and have the following format,\n    <numerator>/<denominator> where both numerator and denominator are positive whole numbers.\n\n    You can assume that x, and n are valid fractions, and do not have zero as denominator.\n\n    simplify("1/5", "5/1") = True\n    simplify("1/6", "2/1") = False\n    simplify("7/10", "10/2") = False\n    """\n'},
  'outputs': {'response_code': 'def check(candidate):\n\n    # Check some simple cases\n    assert candidate("1/5", "5/1") == True, \'test1\'\n    assert candidate("1/6", "2/1") == False, \'test2\'\n    assert candidate("5/1", "3/1") == True, \'test3\'\n    assert candidate("7/10", "10/2") == False, \'test4\'\n    assert candidate("2/10", "50/10") == True, \'test5\'\

In [5]:
dataset = client_langsmith.list_datasets()

### Avaliando o agente

In [6]:
from langsmith import traceable
from langchain_core.messages import HumanMessage
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact


code_agent = CodeAgentReact(model="qwen/qwen3-next-80b-a3b-instruct", model_provider="nvidia")

agent = code_agent.create_agent()


INFO:code_agent.creat_react_code_agent.code_agent_react:CodeAgentReact inicializado: provider=nvidia, checkpointer=False
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:229: UserWarning: Found qwen/qwen3-next-80b-a3b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Modelo provider=nvidia inicializado com sucesso
c:\Users\jefer\Documents\Ciencia-de-dados\LLMs\AgenteCodificaoLangGraph\.venv\Lib\site-packages\langchain_nvidia_ai_endpoints\chat_models.py:715: UserWarning: Model 'qwen/qwen3-next-80b-a3b-instruct' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
INFO:code_agent.creat_react_code_agent.code_agent_react:Agente React criado com sucesso


In [7]:
@traceable
async def agent_avaliado(question):
    answer = await agent.ainvoke(
    {
        "messages": 
            [HumanMessage(role="user",
                          content=question)],
        "todos": [],
    }
)
    return answer['messages'][-1].content


In [8]:
# We'll first define a custom code evaluator, which are useful to measure deterministic or close-ended metrics.
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200 

LLM-as-a-Judge Evaluator
For open-ended metrics, it's can be powerful to use an LLM to score the outputs.

Let's use an LLM to check whether our application produces correct outputs. First, let's define a scoring schema for our LLM to adhere to in its response.

In [9]:
from pydantic import BaseModel, Field

# Define a scoring schema that our LLM must adhere to
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score for the correctness of the answer, of an answer between 0 and 1")

In [10]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

async def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

    <Rubric>
        A correct answer:
        - Provides accurate information
        - Uses suitable analogies and examples
        - Contains no factual errors
        - Is logically consistent

        When scoring, you should penalize:
        - Factual errors
        - Incoherent analogies and examples
        - Logical inconsistencies
    </Rubric>

    <Instructions>
        - Carefully read the input and output
        - Use the reference output to determine if the model output contains errors
        - Focus whether the model output uses accurate analogies and is logically consistent
    </Instructions>

    <Reminder>
        The analogies in the output do not need to match the reference output exactly. Focus on logical consistency.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Use the reference outputs below to help you evaluate the correctness of the response:
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["aswer_code"], outputs["output"], reference_outputs["response_code"])

    
    """model = init_chat_model(model = "moonshotai/kimi-k2-instruct-0905", model_provider = "nvidia")
    
    model_stutured = model.with_structured_output(CorrectnessScore)
    
    response = await model_stutured.ainvoke([HumanMessage(content=prompt)])"""
    
    model = GoogleModel('models/gemini-2.0-flash-lite')
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = agent.run_sync(prompt)
    except Exception as e:
        logger.error(f"Erro do tipo: {e}")
        model = GoogleModel('models/gemini-2.0-flash')
        agent = Agent(model, output_type=CorrectnessScore)
        response = agent.run_sync(prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response

## Code Correctness 

In [11]:
CODE_CORRECTNESS_PROMPT_WITH_REFERENCE_OUTPUTS = """You are an expert code reviewer evaluating code for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct code solution:
  - Solves the problem completely as specified in the input
  - Should contain only valid code without any additional text
  - Handles all edge cases appropriately
  - Contains absolutely no bugs or logical errors
  - Uses efficient and appropriate algorithms/data structures
  - Follows language-specific best practices
  - Has correct syntax and would compile/run without errors

  When scoring, you should penalize:
  - Logical errors or bugs that would cause incorrect behavior
  - Missing edge case handling
  - Overly inefficient implementations when better approaches exist
  - Incomplete solutions that don't address all requirements
  - Syntax errors that would prevent compilation/execution
  - Security vulnerabilities or unsafe practices
  - Additional text that is not code
</Rubric>

<Instructions>
  - Carefully analyze both the output code and the initial input query
  - Meticulously check for functional correctness and completeness
  - Focus on whether the code would work correctly rather than style preferences
  - Compare the output with the reference output to verify correctness
  - The reference output represents the expected behavior or result
  - Code that produces results matching the reference output should be scored higher
  - Consider edge cases where the code might produce correct results for the given examples but fail in other scenarios
</Instructions>

<Reminder>
  The goal is to evaluate whether the code correctly solves the given problem and produces output that matches the reference.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_output>
{reference_outputs}
</reference_output>
"""

In [12]:
from pydantic import BaseModel, Field
from code_agent.get_routem_llm.routem_llm import LlmRouter
from typing import Dict, Any
class CodeOutput(BaseModel):
    """Schema for code solutions to questions about LCEL."""
    code: str = Field(description="You should stract the code solution from the response.")

async def extract_code_from_response(response: str) -> str:
    """Extract code solution from the model response."""
    router_structured = LlmRouter(response, CodeOutput)  # type:ignore

    response_code_formatted = await router_structured.llm_router()
    
    return response_code_formatted['code']

async def correctness_code(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    
    output_formatted = await extract_code_from_response(outputs["output"])
    
    prompt = CODE_CORRECTNESS_PROMPT_WITH_REFERENCE_OUTPUTS.format(inputs["aswer_code"], output_formatted, reference_outputs["response_code"])

    
    model = GoogleModel('models/gemini-2.0-flash-lite')
    agent = Agent(model, output_type=CorrectnessScore)

    try:
        response = agent.run_sync(prompt)
    except Exception as e:
        logger.error(f"Erro do tipo: {e}")
        model = GoogleModel('models/gemini-2.0-flash')
        agent = Agent(model, output_type=CorrectnessScore)
        response = agent.run_sync(prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response



In [13]:
import nest_asyncio

# Apply nest_asyncio at the start of your notebook
nest_asyncio.apply()

def correctness_sync(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    return asyncio.run(correctness(inputs, outputs, reference_outputs))

In [14]:
response = correctness_sync({"aswer_code": "def add(a, b):\n    return a + b"}, {"output": "def add(a, b):\n    return a + b"}, {"response_code": "def add(a, b):\n    return a + b"})

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent "HTTP/1.1 200 OK"


In [15]:
# 4. Define a function to run your application
async def run(inputs: dict):
    return await agent_avaliado(inputs["aswer_code"])


In [16]:
experiments_realizados = [experiment.name.split('-aleatorios-')[1].rsplit('-', 2)[0] for experiment in client_langsmith.list_projects() if "Human-Eval-Code-5" in experiment.name]


In [17]:
"""from langsmith import evaluate, aevaluate

results = asyncio.run(aevaluate(
                run,
                data=dataset_name,
                evaluators=[correctness_code, correctness, conciseness],
                experiment_prefix=f"{dataset_name}-",))"""

'from langsmith import evaluate, aevaluate\n\nresults = asyncio.run(aevaluate(\n                run,\n                data=dataset_name,\n                evaluators=[correctness_code, correctness, conciseness],\n                experiment_prefix=f"{dataset_name}-",))'

In [18]:
from langsmith import evaluate, aevaluate
rodar = False

if rodar:
    modelos = [
        {"model": "openai/gpt-oss-20b",  "provider": "groq"},
        {"model": "meta-llama/llama-4-scout-17b-16e-instruct",  "provider": "groq"},
        {"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "groq"},
        {"model": "qwen/qwen3-next-80b-a3b-instruct",  "provider": "nvidia"},
        {"model": "deepseek-ai/deepseek-v3.1",  "provider": "nvidia"},
        {"model": "microsoft/phi-4-mini-instruct",  "provider": "nvidia"},
        {"model": "moonshotai/kimi-k2-instruct-0905",  "provider": "nvidia"},
        {"model": "meta/llama-4-scout-17b-16e-instruct",  "provider": "nvidia"},
        {"model": "openai/gpt-oss-120b",  "provider": "nvidia"},
        {"model": "moonshotai/kimi-k2-instruct",  "provider": "nvidia"},
        {"model": "meta/llama-3.3-70b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.3-nemotron-super-49b-v1.5",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.1-nemotron-ultra-253b-v1",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-405b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.1-nemotron-nano-4b-v1.1",  "provider": "nvidia"},
        {"model": "ibm/granite-3.3-8b-instruct",  "provider": "nvidia"},
        {"model": "nv-mistralai/mistral-nemo-12b-instruct",  "provider": "nvidia"},
        {"model": "nvidia/llama-3.3-nemotron-super-49b-v1",  "provider": "nvidia"},
        {"model": "mistralai/mistral-small-3.1-24b-instruct-2503",  "provider": "nvidia"},
        {"model": "qwen/qwq-32b",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-8b-instruct",  "provider": "nvidia"},
        {"model": "mistralai/mistral-nemotron",  "provider": "nvidia"},
        {"model": "meta/llama-3.2-3b-instruct",  "provider": "nvidia"},
        {"model": "openai/gpt-oss-20b",  "provider": "nvidia"},
        {"model": "mistralai/mistral-large-2-instruct",  "provider": "nvidia"},
        {"model": "deepseek-ai/deepseek-r1-0528",  "provider": "nvidia"},
        {"model": "meta/llama-3.1-70b-instruct",  "provider": "nvidia"}
        
        
    ]

    for modelo in modelos:
        print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provider']}")
        
        if modelo['model'] in experiments_realizados:
            print(f"Modelo {modelo['model']} já concluído")
            continue
        else:
            code_agent = CodeAgentReact(model=modelo['model'], model_provider=modelo['provider'])
            agent = code_agent.create_agent()
            
            @traceable
            async def agent_avaliado(question):
                answer = await agent.ainvoke(
                {
                    "messages": 
                        [HumanMessage(role="user",
                                    content=question)],
                    "todos": [],
                }
            )
                return answer['messages'][-1].content
            
            async def run(inputs: dict):
                return await agent_avaliado(inputs["aswer_code"])
            
            
            results = asyncio.run(aevaluate(
                run,
                data=dataset_name,
                evaluators=[correctness_code,correctness, conciseness],
                experiment_prefix=f"{dataset_name}-{modelo['model']}-{modelo['provider']}"))
            
            #models_concluidos.append(modelo['model']) 

### Avaliando os resultados

In [19]:
from langsmith import Client
client_langsmith = Client()

In [20]:
experiments_names = [experiment.name for experiment in client_langsmith.list_projects() if "Human-Eval-Code-" in experiment.name]
experiments_names

['Human-Eval-Code-15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb',
 'Human-Eval-Code-15-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-c66a1c00',
 'Human-Eval-Code-15-aleatorios-openai/gpt-oss-20b-groq-037d534d',
 'Human-Eval-Code-15-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-f0050272',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-0e1a2884',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-18224147',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-50f6fdb0',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-ab726e5d',
 'Human-Eval-Code-15-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-bf877a7a',
 'Human-Eval-Code-15-aleatorios-meta/llama-3.1-8b-instruct-nvidia-e63a2c52',
 'Human-Eval-Code-15-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7681265d']

In [21]:
# Set this to load expt results
for experiment_name in experiments_names:
    print("Loading results for experiment:", experiment_name)
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", experiment_results.latency_p50)
    print("Latency p99:", experiment_results.latency_p99)
    print("Token Usage:", experiment_results.total_tokens)
    print("Feedback Stats:", experiment_results.feedback_stats)
    print("*" * 50)

Loading results for experiment: Human-Eval-Code-15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb
Latency p50: 0:01:41.142500
Latency p99: 0:01:58.616250
Token Usage: 155942
Feedback Stats: {'conciseness': {'n': 4, 'avg': 0.75, 'stdev': 0.4330127018922193, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}, 'correctness': {'n': 4, 'avg': 1.0, 'stdev': 0.0, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}}
**************************************************
Loading results for experiment: Human-Eval-Code-15-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-c66a1c00
Latency p50: 0:00:52.491000
Latency p99: 0:00:52.491000
Token Usage: 7041
Feedback Stats: None
**************************************************
Loading results for experiment: Human-Eval-Code-15-aleatorios-openai/gpt-oss-20b-groq-037d534d
Latency p50: 0:00:27.287000
Latency p99: 0:00:27.287000
Token Usage: 6486
Feedback Stats: None
********************

In [22]:
import pandas as pd
data_frames = pd.DataFrame()
for experiment_name in experiments_names:
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    name_column = experiment_name.split("Human-Eval-Code-")[1]
    data = pd.DataFrame.from_dict(experiment_results.dict()).T
    try:
        data_frames[name_column] = data["correctness"]
    except Exception as e:
        print(e)
        pass
    

'correctness'
'correctness'


In [23]:
data_frames

,15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb,15-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-f0050272,15-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-0e1a2884,15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-18224147,15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-50f6fdb0,15-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-ab726e5d,15-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-bf877a7a,15-aleatorios-meta/llama-3.1-8b-instruct-nvidia-e63a2c52,15-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7681265d
id,91bb777f-76f5-4c7a-8032-3084662cab51,54808385-ccb5-4124-94f7-135e9f780e0f,57758c2b-9ac9-4a9e-8fe5-7eb5d95e8cba,7035833f-b7fb-42b1-96d8-7b27fad27d9c,28fbdadd-4561-484e-a659-0e06439a366f,256d9cbe-03e4-4962-a2cc-4810ca1a6a97,475db5b6-5541-4b0c-9609-dbd6ff39585d,73c77181-227c-4dfa-bd3c-0bbd849dd68e,fb58b37f-ef3d-473c-ad9a-114f89fa3907
start_time,2025-10-20 19:22:41.762300+00:00,2025-10-20 16:21:05.692438+00:00,2025-10-20 14:59:45.864521+00:00,2025-10-20 14:53:42.304954+00:00,2025-10-20 14:47:17.522306+00:00,2025-10-20 13:46:24.754840+00:00,2025-10-20 13:22:06.750276+00:00,2025-10-20 12:57:35.702511+00:00,2025-10-20 12:31:31.239488+00:00
end_time,None,None,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None,None,None
name,Human-Eval-Code-15-aleatorios-moonshotai/kimi-...,Human-Eval-Code-15-aleatorios-deepseek-ai/deep...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.3...,Human-Eval-Code-15-aleatorios-mistralai/mistra...,Human-Eval-Code-15-aleatorios-meta/llama-3.1-8...,Human-Eval-Code-15-aleatorios-meta/llama-3.1-7...
extra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenant_id,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87
reference_dataset_id,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87
run_count,4,12,15,15,15,15,15,15,15
latency_p50,0 days 00:01:41.142500,0 days 00:06:09.495000,0 days 00:00:31.103000,0 days 00:00:19.293000,0 days 00:00:19.335000,0 days 00:01:45.845000,0 days 00:00:28.942000,0 days 00:01:20.111000,0 days 00:01:44.631000


#### Selecionando os melhores modelos e Testando com mais exemplos

In [24]:
melhores_experiments = [colunas for colunas in data_frames.columns if data_frames[colunas]['feedback_stats']['avg'] > 0.55]
data_frames_melhores = data_frames[melhores_experiments]
data_frames_melhores.head()

,15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb,15-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-f0050272,15-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-0e1a2884,15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-18224147,15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-50f6fdb0,15-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-bf877a7a,15-aleatorios-meta/llama-3.1-8b-instruct-nvidia-e63a2c52
id,91bb777f-76f5-4c7a-8032-3084662cab51,54808385-ccb5-4124-94f7-135e9f780e0f,57758c2b-9ac9-4a9e-8fe5-7eb5d95e8cba,7035833f-b7fb-42b1-96d8-7b27fad27d9c,28fbdadd-4561-484e-a659-0e06439a366f,475db5b6-5541-4b0c-9609-dbd6ff39585d,73c77181-227c-4dfa-bd3c-0bbd849dd68e
start_time,2025-10-20 19:22:41.762300+00:00,2025-10-20 16:21:05.692438+00:00,2025-10-20 14:59:45.864521+00:00,2025-10-20 14:53:42.304954+00:00,2025-10-20 14:47:17.522306+00:00,2025-10-20 13:22:06.750276+00:00,2025-10-20 12:57:35.702511+00:00
end_time,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None
name,Human-Eval-Code-15-aleatorios-moonshotai/kimi-...,Human-Eval-Code-15-aleatorios-deepseek-ai/deep...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-mistralai/mistra...,Human-Eval-Code-15-aleatorios-meta/llama-3.1-8...


In [25]:
melhores_modelos = [{"model":coluna.split('-aleatorios-')[1].rsplit('-', 2)[0], "provedor":coluna.split('-')[-2]} for coluna in data_frames_melhores.columns]

In [26]:
melhores_modelos

[{'model': 'moonshotai/kimi-k2-instruct-0905', 'provedor': 'groq'},
 {'model': 'deepseek-ai/deepseek-v3.1', 'provedor': 'nvidia'},
 {'model': 'nvidia/llama-3.1-nemotron-ultra-253b-v1', 'provedor': 'nvidia'},
 {'model': 'nvidia/llama-3.1-nemotron-nano-4b-v1.1', 'provedor': 'nvidia'},
 {'model': 'nvidia/llama-3.1-nemotron-nano-4b-v1.1', 'provedor': 'nvidia'},
 {'model': 'mistralai/mistral-small-3.1-24b-instruct-2503',
  'provedor': 'nvidia'},
 {'model': 'meta/llama-3.1-8b-instruct', 'provedor': 'nvidia'}]

In [27]:
from datasets import load_dataset
from tqdm import tqdm

human_eval = load_dataset("openai_humaneval")['test']

data_set_code_langsmith = []
for problem in tqdm(human_eval, desc="Problems", unit="problem"):
    prompt = problem['prompt']
    test_code = problem['test']
    
    dict_data = {
      "inputs": {"aswer_code": prompt},
      "outputs": {"response_code": test_code},
  }
    
   
    data_set_code_langsmith.append(dict_data)

Problems: 100%|██████████| 164/164 [00:00<00:00, 549.94problem/s]


In [28]:
from langsmith import Client

client_langsmith = Client()


tamanho_amostra = 15

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

import random
data_aleatorios = []
index_aleatorios = random.sample(range(len(data_set_code_langsmith)), tamanho_amostra)
for index in index_aleatorios:
    data_aleatorios.append(data_set_code_langsmith[index])


# Create dataset if it doesn't exist
if not client_langsmith.has_dataset(dataset_name=dataset_name):
    dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name, 
        description="The HumanEval dataset released by OpenAI includes 164 programming problems with a function sig- nature, docstring, body, and several unit tests. They were handwritten to ensure not to be included in the training set of code generation models."
    )
    
    client_langsmith.create_examples(dataset_id=dataset.id, examples=data_aleatorios)

In [29]:
experiments_realizados = [experiment.name.split('-aleatorios-')[1].rsplit('-', 2)[0] for experiment in client_langsmith.list_projects() if "Human-Eval-Code-15" in experiment.name]

experiments_realizados

['moonshotai/kimi-k2-instruct-0905',
 'qwen/qwen3-next-80b-a3b-instruct',
 'openai/gpt-oss-20b',
 'deepseek-ai/deepseek-v3.1',
 'nvidia/llama-3.1-nemotron-ultra-253b-v1',
 'nvidia/llama-3.1-nemotron-nano-4b-v1.1',
 'nvidia/llama-3.1-nemotron-nano-4b-v1.1',
 'nvidia/llama-3.3-nemotron-super-49b-v1',
 'mistralai/mistral-small-3.1-24b-instruct-2503',
 'meta/llama-3.1-8b-instruct',
 'meta/llama-3.1-70b-instruct']

In [30]:
models_concluidos_melhores = []

rodar = False
if rodar:
    for modelo in melhores_modelos:
        print(f"Rodando o agente com o modelo {modelo['model']} do provedor {modelo['provedor']}")
        
        if modelo['model'] in experiments_realizados:
            print("Modelo já concluído")
            continue
        else:
            code_agent = CodeAgentReact(model=modelo['model'], model_provider=modelo['provedor'])
            agent = code_agent.create_agent()
            
            @traceable
            async def agent_avaliado(question):
                answer = await agent.ainvoke(
                {
                    "messages": 
                        [HumanMessage(role="user",
                                    content=question)],
                    "todos": [],
                }
            )
                return answer['messages'][-1].content
            
            async def run(inputs: dict):
                return await agent_avaliado(inputs["aswer_code"])
            
            
            results = asyncio.run(aevaluate(
                run,
                data=dataset_name,
                evaluators=[correctness, conciseness],
                experiment_prefix=f"{dataset_name}-{modelo['model']}-{modelo['provedor']}"))
            
            models_concluidos_melhores.append(modelo['model']) 

# Avaliando os 

In [31]:
experiments_names = [experiment.name for experiment in client_langsmith.list_projects() if "Human-Eval-Code-15" in experiment.name]
experiments_names

['Human-Eval-Code-15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb',
 'Human-Eval-Code-15-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-c66a1c00',
 'Human-Eval-Code-15-aleatorios-openai/gpt-oss-20b-groq-037d534d',
 'Human-Eval-Code-15-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-f0050272',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-0e1a2884',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-18224147',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-50f6fdb0',
 'Human-Eval-Code-15-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-ab726e5d',
 'Human-Eval-Code-15-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-bf877a7a',
 'Human-Eval-Code-15-aleatorios-meta/llama-3.1-8b-instruct-nvidia-e63a2c52',
 'Human-Eval-Code-15-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7681265d']

In [32]:
# Set this to load expt results
for experiment_name in experiments_names:
    print("Loading results for experiment:", experiment_name)
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", experiment_results.latency_p50)
    print("Latency p99:", experiment_results.latency_p99)
    print("Token Usage:", experiment_results.total_tokens)
    print("Feedback Stats:", experiment_results.feedback_stats)
    print("*" * 50)

Loading results for experiment: Human-Eval-Code-15-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb
Latency p50: 0:01:41.142500
Latency p99: 0:01:58.616250
Token Usage: 155942
Feedback Stats: {'conciseness': {'n': 4, 'avg': 0.75, 'stdev': 0.4330127018922193, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}, 'correctness': {'n': 4, 'avg': 1.0, 'stdev': 0.0, 'errors': 0, 'values': {}, 'type': 'primary', 'contains_thread_feedback': False}}
**************************************************
Loading results for experiment: Human-Eval-Code-15-aleatorios-qwen/qwen3-next-80b-a3b-instruct-nvidia-c66a1c00
Latency p50: 0:00:52.491000
Latency p99: 0:00:52.491000
Token Usage: 7041
Feedback Stats: None
**************************************************
Loading results for experiment: Human-Eval-Code-15-aleatorios-openai/gpt-oss-20b-groq-037d534d
Latency p50: 0:00:27.287000
Latency p99: 0:00:27.287000
Token Usage: 6486
Feedback Stats: None
********************

In [33]:
import pandas as pd
data_frames = pd.DataFrame()
for experiment_name in experiments_names:
    experiment_results = client_langsmith.read_project(project_name=experiment_name, include_stats=True)
    name_column = experiment_name.split("Human-Eval-Code-15")[1]
    data = pd.DataFrame.from_dict(experiment_results.dict()).T
    try:
        data_frames[name_column] = data["correctness"]
    except Exception as e:
        print(e)
        pass

'correctness'
'correctness'


In [34]:
# Ordena o DataFrame pela coluna 'total_tokens' em ordem decrescente
# Usar sort_values em vez de sort_index para ordenar por valores de coluna
data_frames

,-aleatorios-moonshotai/kimi-k2-instruct-0905-groq-6ee838fb,-aleatorios-deepseek-ai/deepseek-v3.1-nvidia-f0050272,-aleatorios-nvidia/llama-3.1-nemotron-ultra-253b-v1-nvidia-0e1a2884,-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-18224147,-aleatorios-nvidia/llama-3.1-nemotron-nano-4b-v1.1-nvidia-50f6fdb0,-aleatorios-nvidia/llama-3.3-nemotron-super-49b-v1-nvidia-ab726e5d,-aleatorios-mistralai/mistral-small-3.1-24b-instruct-2503-nvidia-bf877a7a,-aleatorios-meta/llama-3.1-8b-instruct-nvidia-e63a2c52,-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7681265d
id,91bb777f-76f5-4c7a-8032-3084662cab51,54808385-ccb5-4124-94f7-135e9f780e0f,57758c2b-9ac9-4a9e-8fe5-7eb5d95e8cba,7035833f-b7fb-42b1-96d8-7b27fad27d9c,28fbdadd-4561-484e-a659-0e06439a366f,256d9cbe-03e4-4962-a2cc-4810ca1a6a97,475db5b6-5541-4b0c-9609-dbd6ff39585d,73c77181-227c-4dfa-bd3c-0bbd849dd68e,fb58b37f-ef3d-473c-ad9a-114f89fa3907
start_time,2025-10-20 19:22:41.762300+00:00,2025-10-20 16:21:05.692438+00:00,2025-10-20 14:59:45.864521+00:00,2025-10-20 14:53:42.304954+00:00,2025-10-20 14:47:17.522306+00:00,2025-10-20 13:46:24.754840+00:00,2025-10-20 13:22:06.750276+00:00,2025-10-20 12:57:35.702511+00:00,2025-10-20 12:31:31.239488+00:00
end_time,None,None,None,None,None,None,None,None,None
description,None,None,None,None,None,None,None,None,None
name,Human-Eval-Code-15-aleatorios-moonshotai/kimi-...,Human-Eval-Code-15-aleatorios-deepseek-ai/deep...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.1...,Human-Eval-Code-15-aleatorios-nvidia/llama-3.3...,Human-Eval-Code-15-aleatorios-mistralai/mistra...,Human-Eval-Code-15-aleatorios-meta/llama-3.1-8...,Human-Eval-Code-15-aleatorios-meta/llama-3.1-7...
extra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenant_id,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87,d9ab5018-45a8-5efd-961d-320c87839c87
reference_dataset_id,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87,3454f0a7-de60-4470-a562-d949a337cd87
run_count,4,12,15,15,15,15,15,15,15
latency_p50,0 days 00:01:41.142500,0 days 00:06:09.495000,0 days 00:00:31.103000,0 days 00:00:19.293000,0 days 00:00:19.335000,0 days 00:01:45.845000,0 days 00:00:28.942000,0 days 00:01:20.111000,0 days 00:01:44.631000


### Acessando datasets do LangSmith

In [35]:
# List all datasets
from langsmith import Client

client = Client()

datasets = client.list_datasets()
for dataset in datasets:
    print(dataset.id, dataset.name)

b017c54f-76bb-4d21-8ea2-5428b305759d Human-Eval-Code-5-aleatorios
3454f0a7-de60-4470-a562-d949a337cd87 Human-Eval-Code-15-aleatorios
b9dd1aaa-017b-4280-96ca-cd6f094773fa deep_research_supervisor_parallelism
5873a0fe-22fd-4a3d-8eb6-86330c26e52f deep_research_agent_termination
2bdcce58-182b-4a36-98a5-74720bc26a05 deep_research_scoping
27916b23-40e0-4874-8d5c-ddbebfcb8958 E-mail Triage Evaluation
b746e978-0744-44de-9f42-6cbd42bbd1e3 agents-from-scratch.test_response
623cff70-db0b-4993-bfe4-6ee88d1a46ac agents-from-scratch.test_tools
a2defcc0-e281-43c6-b351-c7930a9307ac Financial Advisory RAG Evaluation
77fa7a2a-4ece-4096-96fa-ed9d1d9cff81 Healthcare Agent Trajectory Evaluation
43be1111-6a51-4928-a33e-940a33a8b40a Reasoning and Bias
ba1609ff-70a6-4a64-b849-1892bbd24274 QA Example Dataset
aea62e89-af3d-4bb0-98e6-275fd643bfe5 Sample dataset
da848d07-6fa9-4ffe-a503-570e10840ac0 Insurance Claims


In [37]:
examples = client.list_examples(dataset_id="3454f0a7-de60-4470-a562-d949a337cd87")
for example in examples:
    print(example.inputs, example.outputs)

{'aswer_code': '\n\ndef car_race_collision(n: int):\n    """\n    Imagine a road that\'s a perfectly straight infinitely long line.\n    n cars are driving left to right;  simultaneously, a different set of n cars\n    are driving right to left.   The two sets of cars start out being very far from\n    each other.  All cars move in the same speed.  Two cars are said to collide\n    when a car that\'s moving left to right hits a car that\'s moving right to left.\n    However, the cars are infinitely sturdy and strong; as a result, they continue moving\n    in their trajectory as if they did not collide.\n\n    This function outputs the number of such collisions.\n    """\n'} {'response_code': '\n\nMETADATA = {}\n\n\ndef check(candidate):\n    assert candidate(2) == 4\n    assert candidate(3) == 9\n    assert candidate(4) == 16\n    assert candidate(8) == 64\n    assert candidate(10) == 100\n\n'}
{'aswer_code': '\ndef f(n):\n    """ Implement the function f that takes n as a parameter,\n

In [38]:
experiment_results.dict()

{'id': UUID('fb58b37f-ef3d-473c-ad9a-114f89fa3907'),
 'start_time': datetime.datetime(2025, 10, 20, 12, 31, 31, 239488, tzinfo=datetime.timezone.utc),
 'end_time': None,
 'description': None,
 'name': 'Human-Eval-Code-15-aleatorios-meta/llama-3.1-70b-instruct-nvidia-7681265d',
 'extra': {'metadata': {'git': {'tags': None,
    'dirty': True,
    'branch': 'secundario',
    'commit': '9343b37541f45c0533b9c2d3c75328ffe8fe663e',
    'repo_name': 'AgenteCodificaoLangGraph',
    'remote_url': 'https://github.com/Jeferson100/Code-Agent.git',
    'author_name': 'JEFERSON DIONEI SEHNEM',
    'commit_time': '1760556298',
    'author_email': 'sehnemjeferson@gmail.com'},
   'revision_id': '9343b37-dirty',
   'dataset_splits': ['base'],
   'dataset_version': '2025-10-20T12:30:03.970797+00:00',
   'num_repetitions': 1}},
 'tenant_id': UUID('d9ab5018-45a8-5efd-961d-320c87839c87'),
 'reference_dataset_id': UUID('3454f0a7-de60-4470-a562-d949a337cd87'),
 'run_count': 15,
 'latency_p50': datetime.timedel

In [51]:
TRAJECTORY_ACCURACY_PROMPT = """
You are an **expert evaluator of AI agent trajectories**.
Your goal is to assess whether the agent correctly and efficiently used the available tools
to achieve the task’s goal.

<Context>
The agent has access to the following tools and limits:

1. **write_todos** — Manage or update the TODO list (max 3 calls per session).
2. **read_todos** — Read the TODO list for context (max 5 calls per session).
3. **web_search** — Retrieve verified external information.
   - Simple tasks → max 1 call
   - Normal tasks → max 2 calls
   - Complex/research-heavy tasks → max 3 calls
   - Never exceed 3 total web_search calls.
4. **think_tool** — Reflect after each search or major step (mandatory after every web_search).
   - Limit: equal to (number of web_search calls + 2)
5. **write_code** — Produce or refine code (max 3 calls per coding task).
</Context>

<Rubric>
An **accurate trajectory**:
- Uses the correct tools for the correct purposes.
- Respects the call limits for each tool.
- Demonstrates logical sequencing (e.g., search → reflection → code → explanation).
- Includes a think_tool call immediately after each web_search.
- Avoids redundant, unnecessary, or missing tool calls.
- Shows clear reasoning progression toward the goal.
- Is efficient, though not necessarily perfectly optimal.
</Rubric>

<Evaluation Process>
1. Infer the user’s overall goal from the trajectory’s first and last messages.
2. Examine each step to verify:
   - Was the chosen tool appropriate for that stage?
   - Were calls made in the correct logical order?
   - Were tool limits respected?
   - Were required reflections (think_tool) present after searches?
3. Identify any misuse (wrong tool, skipped reflection, or exceeding limits).
4. Consider the overall flow — does it make sense and align with the intended goal?

<Output Format>
Provide your evaluation in this format:

**Goal Understanding:**  
<Your brief summary of what the agent was trying to achieve.>

**Tool Usage Accuracy (0.0–0.1):**  
Score based on how well the agent chose and sequenced tools.  
- 10 = perfect tool use, clear purpose, within limits.  
- 7–9 = minor inefficiencies or redundant steps.  
- 4–6 = occasional misuse or missing mandatory steps.  
- 1–3 = repeated misuse, skipped required reflections, or exceeded limits.  
- 0 = nonsensical or rule-breaking tool use.

**Reasoning Flow (0.0–0.1):**  
Score for logical coherence, step progression, and consistency.  
- 10 = clear, rational sequence from start to goal.  
- 7–9 = mostly coherent with small lapses.  
- 4–6 = disjointed or missing reasoning links.  
- 0–3 = chaotic or unrelated steps.

**Efficiency (0.0–0.5):**  
Score for minimizing redundant tool calls or unnecessary steps.

**Final Grade (0.0–0.25):**  
Sum of the three scores above.

**Comments:**  
- Mention any specific strengths or weaknesses.  
- Highlight incorrect or missing tool calls.  
- Note any violations of tool call limits or missing reflections.

<trajectory>
{outputs}
</trajectory>
"""


In [52]:
#from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.huggingface import HuggingFaceModel
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

class TrajectoryScore(BaseModel):
    """Avaliation score for the trajectory."""
    score: int = Field(description="Score between 0 and 1")

async def trajectory_accuracy(outputs: dict) -> bool:
    
    model = GoogleModel('models/gemini-2.0-flash-lite')
    agent = Agent(model, output_type=TrajectoryScore)

    prompt = TRAJECTORY_ACCURACY_PROMPT.format(outputs=outputs)

    try:
        response = agent.run_sync(prompt)
    except Exception as e:
        logger.error(f"Erro do tipo: {e}")
        model = GoogleModel('models/gemini-2.0-flash')
        agent = Agent(model, output_type=TrajectoryScore)
        response = agent.run_sync(prompt)
    
    if isinstance(response, dict):
        response = response["score"]
    else:
        response = response.output.score
    return response

In [ ]:
#from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from agentevals.trajectory.llm import create_trajectory_llm_as_judge #, TRAJECTORY_ACCURACY_PROMPT
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langsmith import traceable
from langchain_core.messages import HumanMessage
from code_agent.creat_react_code_agent.code_agent_react import CodeAgentReact

code_agent = CodeAgentReact(
    model="meta/llama-4-scout-17b-16e-instruct",
    #model="qwen/qwen3-next-80b-a3b-instruct",
    model_provider="nvidia")

agent = code_agent.create_agent()

"""evaluator = create_trajectory_llm_as_judge(  
    model="groq:qwen/qwen3-32b",
    prompt=TRAJECTORY_ACCURACY_PROMPT,  
)"""  

def run(inputs: dict):
    result = agent.invoke({
        "messages": [HumanMessage(content=inputs["aswer_code"])],
    })

    return result["messages"]

In [ ]:

async def run(inputs: dict):
    result = await agent.ainvoke(
        {
            "messages": 
                [HumanMessage(role="user",
                            content=inputs["aswer_code"])],
            "todos": [],
        }
    )
    return result["messages"]
    
tamanho_amostra = 5

dataset_name = f"Human-Eval-Code-{tamanho_amostra}-aleatorios"

results = asyncio.run(aevaluate(
            run,
            data=dataset_name,
            evaluators=[trajectory_accuracy],
            experiment_prefix="test-tools"))